
## План анализа

В ходе EDA будут выполнены следующие шаги:

1. Загрузка и первичный осмотр данных  
2. Анализ пропусков и типов данных  
3. Анализ целевой переменной  
4. Исследование числовых признаков  
5. Анализ категориальных признаков  
6. Географический анализ  
7. Поиск выбросов и аномалий  
8. Анализ корреляций  
9. Формирование выводов  

Импорт нужных библиотек

In [358]:
import pandas as pd
import numpy as np
import plotly.express as px
import scipy

Загрузка данных

In [359]:
df_train = pd.read_csv('../data/train.csv', index_col='id')

Выведим первые 5 записей

In [360]:
df_train.head(5)

,timestamp,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,...,cafe_count_5000_price_2500,cafe_count_5000_price_4000,cafe_count_5000_price_high,big_church_count_5000,church_count_5000,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,price_doc
id,,,,,,,,,,,,,,,,,,,,,
1,2011-08-20,43,27.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,...,9,4,0,13,22,1,0,52,4,5850000
2,2011-08-23,34,19.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,...,15,3,0,15,29,1,10,66,14,6000000
3,2011-08-27,43,29.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,10,3,0,11,27,0,4,67,10,5700000
4,2011-09-01,89,50.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,...,11,2,1,4,4,0,0,26,3,13100000
5,2011-09-05,77,77.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,...,319,108,17,135,236,2,91,195,14,16331452


In [361]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30471 entries, 1 to 30473
Columns: 291 entries, timestamp to price_doc
dtypes: float64(119), int64(156), object(16)
memory usage: 67.9+ MB


In [362]:
print(df_train.duplicated().sum())
df_train[df_train.duplicated()]

10


,timestamp,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,...,cafe_count_5000_price_2500,cafe_count_5000_price_4000,cafe_count_5000_price_high,big_church_count_5000,church_count_5000,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,price_doc
id,,,,,,,,,,,,,,,,,,,,,
3362,2012-08-27,59,NaN,6.0,NaN,NaN,NaN,NaN,NaN,NaN,...,4,2,0,3,15,1,0,24,4,4506800
4331,2012-10-22,61,NaN,18.0,NaN,NaN,NaN,NaN,NaN,NaN,...,11,2,1,5,4,0,1,32,5,8248500
6994,2013-04-03,42,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,3,2,0,2,16,1,0,20,4,3444000
8062,2013-05-22,68,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,3,2,0,2,16,1,0,20,4,5406690
8656,2013-06-24,40,NaN,12.0,NaN,NaN,NaN,NaN,NaN,NaN,...,1,0,0,4,6,0,0,4,1,4112800
14007,2014-01-22,46,28.0,1.0,9.0,1.0,1968.0,2.0,5.0,NaN,...,10,1,0,13,15,1,1,61,4,3000000
17407,2014-04-15,134,134.0,1.0,1.0,1.0,0.0,3.0,0.0,1.0,...,0,0,0,0,1,0,0,0,0,5798496
26678,2014-12-17,62,NaN,9.0,17.0,1.0,NaN,2.0,1.0,1.0,...,371,141,26,150,249,2,105,203,13,6552000
28364,2015-03-14,62,NaN,2.0,17.0,1.0,NaN,2.0,1.0,1.0,...,371,141,26,150,249,2,105,203,13,6520500


In [363]:
df_train['timestamp'] = pd.to_datetime(df_train['timestamp'])

In [364]:
df_train.select_dtypes(include='object')

,product_type,sub_area,culture_objects_top_25,thermal_power_plant_raion,incineration_raion,oil_chemistry_raion,radiation_raion,railroad_terminal_raion,big_market_raion,nuclear_reactor_raion,detention_facility_raion,water_1line,big_road1_1line,railroad_1line,ecology
id,,,,,,,,,,,,,,,
1,Investment,Bibirevo,no,no,no,no,no,no,no,no,no,no,no,no,good
2,Investment,Nagatinskij Zaton,yes,no,no,no,no,no,no,no,no,no,no,no,excellent
3,Investment,Tekstil'shhiki,no,no,no,no,yes,no,no,no,no,no,no,no,poor
4,Investment,Mitino,no,no,no,no,no,no,no,no,no,no,no,no,good
5,Investment,Basmannoe,no,no,no,no,yes,yes,no,no,no,no,no,yes,excellent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30469,Investment,Otradnoe,no,no,yes,no,yes,no,no,no,no,no,no,no,good
30470,Investment,Tverskoe,yes,no,no,no,yes,yes,no,no,yes,no,no,no,poor
30471,OwnerOccupier,Poselenie Vnukovskoe,no,no,no,no,no,no,no,no,no,no,no,no,no data


In [365]:
df_train = df_train.drop_duplicates()

In [366]:
numeric_columns = df_train.select_dtypes(include=[np.number]).columns.tolist()

zero_negative_summary = []

for col in numeric_columns:
    zero_count = (df_train[col] == 0).sum()
    neg_count = (df_train[col] < 0).sum()

    zero_negative_summary.append({
        "column": col,
        "zero_count": zero_count,
        "zero_share": zero_count / len(df_train),
        "negative_count": neg_count,
        "negative_share": neg_count / len(df_train),
        "min": df_train[col].min(),
        "median": df_train[col].median(),
        "max": df_train[col].max()
    })

zero_negative_summary = pd.DataFrame(zero_negative_summary).sort_values("zero_count", ascending=False)
zero_negative_summary

,column,zero_count,zero_share,negative_count,negative_share,min,median,max
155,mosque_count_500,30312,0.995108,0,0.0,0.00,0.00,1.000000e+00
178,mosque_count_1000,29877,0.980828,0,0.0,0.00,0.00,1.000000e+00
152,cafe_count_500_price_high,29627,0.972621,0,0.0,0.00,0.00,3.000000e+00
201,mosque_count_1500,29309,0.962181,0,0.0,0.00,0.00,1.000000e+00
175,cafe_count_1000_price_high,29100,0.955320,0,0.0,0.00,0.00,7.000000e+00
...,...,...,...,...,...,...,...,...
28,full_all,0,0.000000,0,0.0,2546.00,85219.00,1.716730e+06
252,prom_part_5000,0,0.000000,0,0.0,0.21,8.98,2.856000e+01
251,green_part_5000,0,0.000000,0,0.0,3.52,19.76,7.546000e+01
32,young_male,0,0.000000,0,0.0,189.00,5470.00,2.097700e+04


In [367]:
zero_negative_summary.describe()

,zero_count,zero_share,negative_count,negative_share,min,median,max
count,275.000000,275.000000,275.0,275.0,2.750000e+02,2.750000e+02,2.750000e+02
mean,6303.047273,0.206922,0.0,0.0,8.022663e+03,7.101290e+04,1.399960e+06
std,9033.692531,0.296566,0.0,0.0,1.256443e+05,7.396747e+05,1.416090e+07
min,0.000000,0.000000,0.0,0.0,0.000000e+00,0.000000e+00,5.218671e-01
25%,0.000000,0.000000,0.0,0.0,0.000000e+00,1.000000e+00,3.350000e+01
50%,678.000000,0.022258,0.0,0.0,0.000000e+00,5.000000e+00,9.100000e+01
75%,10131.500000,0.332606,0.0,0.0,2.919749e-01,6.900000e+01,1.854165e+03
max,30312.000000,0.995108,0.0,0.0,2.081628e+06,1.050803e+07,2.060718e+08


In [368]:
zero_negative_summary[zero_negative_summary['column'].isin(['price_doc', 'full_sq', 'life_sq', 'floor', 'max_floor', 'build_year', 'num_room', 'kitch_sq', 'state', 'raion_popul'])]

,column,zero_count,zero_share,negative_count,negative_share,min,median,max
7,kitch_sq,1380,0.045304,0,0.0,0.0,6.0,2014.0
3,max_floor,550,0.018056,0,0.0,0.0,12.0,117.0
5,build_year,529,0.017366,0,0.0,0.0,1979.0,20052009.0
1,life_sq,45,0.001477,0,0.0,0.0,30.0,7478.0
6,num_room,14,0.000460,0,0.0,0.0,2.0,19.0
2,floor,9,0.000295,0,0.0,0.0,7.0,77.0
0,full_sq,2,0.000066,0,0.0,0.0,49.0,5326.0
8,state,0,0.000000,0,0.0,1.0,2.0,33.0
10,raion_popul,0,0.000000,0,0.0,2546.0,83502.0,247469.0
274,price_doc,0,0.000000,0,0.0,100000.0,6275947.0,111111112.0


In [369]:
for i in ['price_doc', 'full_sq', 'life_sq', 'floor', 'max_floor', 'build_year', 'num_room', 'state', 'raion_popul']:
    df_train.loc[df_train[i] == 0, i] = np.nan

df_train.loc[df_train['max_floor'] < df_train['floor'], ['max_floor', 'floor']] = np.nan
df_train.loc[df_train['life_sq'] > df_train['full_sq'], ['life_sq', 'full_sq']] = np.nan
df_train.loc[df_train['kitch_sq'] > df_train['full_sq'], ['kitch_sq', 'full_sq']] = np.nan
df_train.loc[df_train['kitch_sq'] > df_train['life_sq'], ['kitch_sq', 'life_sq']] = np.nan
df_train.loc[df_train['kitch_sq'] > df_train['life_sq'], ['kitch_sq', 'life_sq']] = np.nan
df_train.loc[(df_train['build_year'] > 2015) | (df_train['build_year'] < 1800), ['build_year']] = np.nan

In [370]:
df_train.loc[(df_train['full_sq'] > 1000) | (df_train['full_sq'] < 10)]

,timestamp,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,...,cafe_count_5000_price_2500,cafe_count_5000_price_4000,cafe_count_5000_price_high,big_church_count_5000,church_count_5000,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,price_doc
id,,,,,,,,,,,,,,,,,,,,,
3530,2012-09-07,5326.0,22.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,...,7,2,0,5,16,0,2,43,6,6868818.0
6115,2013-02-22,6.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,1,7,1,0,12,1,5177040.0
16292,2014-03-20,1.0,1.0,1.0,1.0,4.0,NaN,1.0,1.0,3.0,...,0,0,0,1,7,1,0,12,1,4457400.0
16741,2014-03-31,1.0,1.0,1.0,1.0,4.0,NaN,1.0,1.0,3.0,...,0,0,0,1,7,1,0,12,1,7820575.0
17197,2014-04-09,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,1.0,...,23,4,1,11,14,0,1,72,9,13066000.0
18603,2014-05-19,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,3.0,...,371,141,26,150,249,2,105,203,13,6675730.0
22174,2014-09-03,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,1.0,...,1,1,0,2,12,0,1,10,0,6256186.0
22725,2014-09-20,1.0,1.0,1.0,25.0,1.0,2014.0,1.0,1.0,1.0,...,1,0,0,4,6,0,0,5,1,4740000.0
22798,2014-09-23,1.0,1.0,7.0,19.0,1.0,2015.0,3.0,1.0,1.0,...,1,0,0,1,6,0,0,6,1,5591788.0


In [371]:
df_train.loc[(df_train['full_sq'] > 1000) | (df_train['full_sq'] < 10), ['full_sq']] = np.nan

In [372]:
zero_negative_summary_30 = zero_negative_summary.head(30)
fig = px.bar(x=zero_negative_summary_30['zero_count'], y=zero_negative_summary_30['column'], title='Количество нулей по признакам, топ 30')
fig.show()

In [373]:
df_na_sum = df_train.isna().sum().reset_index().rename(columns={0:'NA_sum', 'index':'feature'}).sort_values('NA_sum', ascending=False).head(40)
fig = px.bar(x=df_na_sum['NA_sum'], y=df_na_sum['feature'], title='Количество пропусков по признакам, топ 40')
fig.show()

## Анализ таргета

In [374]:
print(f'Кол во пропусков у таргета {df_train['price_doc'].isna().sum()}')
print(f"Кол во нулей у таргета {(df_train['price_doc']==0).sum()}")


Кол во пропусков у таргета 0
Кол во нулей у таргета 0


In [375]:
df_train['price_doc'].describe()

count    3.046100e+04
mean     7.123676e+06
std      4.780682e+06
min      1.000000e+05
25%      4.740002e+06
50%      6.275947e+06
75%      8.300000e+06
max      1.111111e+08
Name: price_doc, dtype: float64

In [376]:
fig = px.histogram(x=df_train['price_doc'], marginal='box')
fig.show()

In [377]:
fig = px.histogram(x=np.log(df_train['price_doc']), marginal='box')
fig.show()

In [378]:
group_by_month = df_train.groupby(df_train['timestamp'].dt.to_period('M'))['price_doc'].agg(['mean', 'median']).reset_index().sort_values('timestamp')
group_by_month['timestamp'] = group_by_month['timestamp'].dt.to_timestamp()
group_by_month

,timestamp,mean,median
0,2011-08-01,5.850000e+06,5850000.0
1,2011-09-01,6.007330e+06,5200000.0
2,2011-10-01,5.808422e+06,5500000.0
3,2011-11-01,6.115835e+06,5600000.0
4,2011-12-01,5.824306e+06,5450000.0
5,2012-01-01,6.884006e+06,6100000.0
6,2012-02-01,6.807456e+06,5700000.0
7,2012-03-01,6.765834e+06,6175000.0
8,2012-04-01,6.800689e+06,6200000.0
9,2012-05-01,7.226219e+06,6100000.0


In [379]:
fig = px.line(
    group_by_month,
    x='timestamp',
    y=['mean', 'median']
)

fig.show()

In [380]:
group_by_month['median'].autocorr(lag=1)

np.float64(0.9013360714615886)

In [381]:
group_by_month['median_diff'] = group_by_month['median'].pct_change()*100
group_by_month['median_diff'].autocorr(lag=1)

np.float64(-0.2725535970261309)

In [382]:
skew_features = df_train.select_dtypes(include='number').skew().sort_values(ascending=False)
skew_features.head(10)

mosque_count_500                     14.193696
leisure_count_500                    10.202231
public_transport_station_min_walk     9.480988
public_transport_station_km           9.480988
build_count_foam                      9.033937
cafe_count_1000_price_high            8.457916
kindergarten_km                       7.940978
preschool_km                          7.437835
school_km                             7.423896
public_healthcare_km                  7.151846
dtype: float64

### Видно, что после устранения аномалий, большинство численных признаков распределены без сильной ассиметрии

In [383]:
skew_features.describe()

count    275.000000
mean       3.182335
std        2.145475
min       -0.068201
25%        1.214249
50%        2.976311
75%        4.834791
max       14.193696
dtype: float64

In [384]:
num_features = df_train.select_dtypes(include='number').columns.to_list()

outliers = {}

for i in num_features:
    q1 = df_train[i].quantile(0.25)
    q3 = df_train[i].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    left_outliers = (df_train[i] < lower_bound).sum()
    right_outliers = (df_train[i] > upper_bound).sum()

    outliers[i] = {
        "left_outliers": left_outliers,
        "right_outliers": right_outliers,
        "total_outliers": left_outliers + right_outliers
    }

df_outliers = pd.DataFrame.from_dict(outliers, orient='index')
df_outliers.sort_values('total_outliers', ascending=False).head(20)

,left_outliers,right_outliers,total_outliers
trc_sqm_500,0,7599,7599
church_count_500,0,7051,7051
office_count_500,0,6764,6764
office_sqm_500,0,6642,6642
cafe_count_1500_price_4000,0,6422,6422
build_count_frame,0,5572,5572
cafe_count_500_price_2500,0,5471,5471
mosque_count_3000,0,5437,5437
build_count_wood,0,5353,5353
build_count_mix,0,5055,5055


In [385]:
df_train[df_train['church_count_500']==df_train['church_count_500'].max()]['sub_area']

id
15369    Tverskoe
Name: sub_area, dtype: object

In [386]:
corr_matrix = df_train.corr(numeric_only=True)

corr_pairs = (
    corr_matrix
    .unstack()
    .reset_index()
)

corr_pairs.columns = ['feature_1', 'feature_2', 'correlation']

corr_pairs = corr_pairs[
    corr_pairs['feature_1'] != corr_pairs['feature_2']
]


corr_pairs = corr_pairs[
    corr_pairs['correlation'].abs() >= 0.7
]
corr_pairs['pair'] = corr_pairs.apply(
    lambda x: tuple(sorted([x['feature_1'], x['feature_2']])),
    axis=1
)

corr_pairs = corr_pairs.drop_duplicates('pair')

corr_pairs = corr_pairs.sort_values(
    by='correlation',
    ascending=False
)

corr_pairs[['feature_1', 'feature_2', 'correlation']]

,feature_1,feature_2,correlation
24841,public_transport_station_km,public_transport_station_min_walk,1.000000
23185,railroad_station_walk_km,railroad_station_walk_min,1.000000
3615,children_preschool,0_6_all,1.000000
4443,children_school,7_14_all,1.000000
20425,metro_min_walk,metro_km_walk,1.000000
...,...,...,...
26930,kremlin_km,trc_count_5000,-0.797344
28597,zd_vokzaly_avto_km,sport_count_5000,-0.805462
26397,sadovoe_km,sport_count_5000,-0.806804
26672,bulvar_ring_km,sport_count_5000,-0.822528


In [387]:
corr_pairs[corr_pairs['correlation']>=0.99][['feature_1', 'feature_2', 'correlation']]

,feature_1,feature_2,correlation
24841,public_transport_station_km,public_transport_station_min_walk,1.000000
23185,railroad_station_walk_km,railroad_station_walk_min,1.000000
3615,children_preschool,0_6_all,1.000000
4443,children_school,7_14_all,1.000000
20425,metro_min_walk,metro_km_walk,1.000000
...,...,...,...
66793,cafe_count_3000_price_2500,cafe_count_3000_price_4000,0.990422
54111,cafe_count_1500_price_2500,cafe_count_2000,0.990416
53561,cafe_count_1500_price_1000,cafe_count_2000,0.990195
72039,cafe_count_5000_na_price,cafe_count_5000_price_1500,0.990166


In [388]:
target_corr = (
    corr_matrix['price_doc']
    .sort_values(ascending=False)
)

target_corr

price_doc             1.000000
full_sq               0.565507
num_room              0.477487
life_sq               0.445866
sport_count_5000      0.294948
                        ...   
ttk_km               -0.272755
bulvar_ring_km       -0.279308
kremlin_km           -0.279406
sadovoe_km           -0.283768
zd_vokzaly_avto_km   -0.284241
Name: price_doc, Length: 275, dtype: float64

### Анализ категориальных признаков

In [389]:
cat_features = df_train.select_dtypes(include='object').columns

nuni = []

for i in cat_features:
    nuni.append(df_train[i].nunique())

cat_df = pd.DataFrame({
    'feature': cat_features,
    'nunique': nuni
})

cat_df


,feature,nunique
0,product_type,2
1,sub_area,146
2,culture_objects_top_25,2
3,thermal_power_plant_raion,2
4,incineration_raion,2
5,oil_chemistry_raion,2
6,radiation_raion,2
7,railroad_terminal_raion,2
8,big_market_raion,2
9,nuclear_reactor_raion,2


In [390]:
df_train[cat_features]

,product_type,sub_area,culture_objects_top_25,thermal_power_plant_raion,incineration_raion,oil_chemistry_raion,radiation_raion,railroad_terminal_raion,big_market_raion,nuclear_reactor_raion,detention_facility_raion,water_1line,big_road1_1line,railroad_1line,ecology
id,,,,,,,,,,,,,,,
1,Investment,Bibirevo,no,no,no,no,no,no,no,no,no,no,no,no,good
2,Investment,Nagatinskij Zaton,yes,no,no,no,no,no,no,no,no,no,no,no,excellent
3,Investment,Tekstil'shhiki,no,no,no,no,yes,no,no,no,no,no,no,no,poor
4,Investment,Mitino,no,no,no,no,no,no,no,no,no,no,no,no,good
5,Investment,Basmannoe,no,no,no,no,yes,yes,no,no,no,no,no,yes,excellent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30469,Investment,Otradnoe,no,no,yes,no,yes,no,no,no,no,no,no,no,good
30470,Investment,Tverskoe,yes,no,no,no,yes,yes,no,no,yes,no,no,no,poor
30471,OwnerOccupier,Poselenie Vnukovskoe,no,no,no,no,no,no,no,no,no,no,no,no,no data


In [391]:
df_train['sub_area'].value_counts()

sub_area
Poselenie Sosenskoe               1772
Nekrasovka                        1610
Poselenie Vnukovskoe              1372
Poselenie Moskovskij               925
Poselenie Voskresenskoe            713
                                  ... 
Molzhaninovskoe                      3
Poselenie Shhapovskoe                2
Poselenie Kievskij                   2
Poselenie Klenovskoe                 1
Poselenie Mihajlovo-Jarcevskoe       1
Name: count, Length: 146, dtype: int64

In [392]:
df_train['product_type'].value_counts()

product_type
Investment       19447
OwnerOccupier    11014
Name: count, dtype: int64

In [393]:
df_train['ecology'].value_counts()

ecology
poor            8018
no data         7651
good            7171
excellent       3936
satisfactory    3685
Name: count, dtype: int64

In [395]:
px.box(x=df_train['ecology'], y=df_train['price_doc'])

In [396]:
px.box(x=df_train['product_type'], y=df_train['price_doc'])

In [402]:
yes_no_features = cat_features.drop(['ecology', 'product_type', 'sub_area'])
df_train[yes_no_features]
summary_df = pd.DataFrame({
    'feature': yes_no_features,

    'yes_rate': [
        (df_train[col] == 'yes').mean()
        for col in yes_no_features
    ],

    'no_rate': [
        (df_train[col] == 'no').mean()
        for col in yes_no_features
    ],

    'median_price_yes': [
        df_train.loc[
            df_train[col] == 'yes',
            'price_doc'
        ].median()
        for col in yes_no_features
    ],

    'median_price_no': [
        df_train.loc[
            df_train[col] == 'no',
            'price_doc'
        ].median()
        for col in yes_no_features
    ]
})

summary_df

,feature,yes_rate,no_rate,median_price_yes,median_price_no
0,culture_objects_top_25,0.063228,0.936772,7400000.0,6200000.0
1,thermal_power_plant_raion,0.054299,0.945701,6600000.0,6250000.0
2,incineration_raion,0.075999,0.924001,5711360.0,6300000.0
3,oil_chemistry_raion,0.009717,0.990283,5850000.0,6291000.0
4,radiation_raion,0.356784,0.643216,6650000.0,6033152.0
5,railroad_terminal_raion,0.037228,0.962772,6573824.0,6260000.0
6,big_market_raion,0.092512,0.907488,5600000.0,6341258.0
7,nuclear_reactor_raion,0.028331,0.971669,6900000.0,6250000.0
8,detention_facility_raion,0.099865,0.900135,6650950.0,6206739.0
9,water_1line,0.076721,0.923279,5678100.0,6300000.0


In [404]:
plot_df = summary_df.melt(
    id_vars='feature',
    value_vars=[
        'median_price_yes',
        'median_price_no'
    ],
    var_name='category',
    value_name='median_price'
)

fig = px.bar(
    plot_df,
    x='feature',
    y='median_price',
    color='category',
    barmode='group'
)

fig.show()

### Отбор важных признаков